# Testes do Loop CreditLLM

Este notebook documenta tres casos do loop descrito em `CONTEXT.md`:

1. Pergunta original do usuario.
2. Limpeza numerica da query.
3. Busca vetorial de contexto/produtos.
4. Chamada da LLM e, quando aplicavel, da ferramenta `validar_elegibilidade`.
5. Resposta final ao usuario.

Os testes usam `OPENAI_API_KEY` e `OPENAI_MODEL` do `.env`.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'sandbox_jupyters' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from app import limpar_query, responder_pergunta

print(f'OPENAI_MODEL: {os.getenv("OPENAI_MODEL", "gpt-5-nano")}')

OPENAI_MODEL: gpt-5-nano


In [2]:
def resumir_resultado(nome, pergunta, esperado_produto=None, exige_tool=True):
    resultado = responder_pergunta(pergunta)
    etapas = [log['etapa'] for log in resultado['logs']]
    ids_contexto = [item['id'] for item in resultado['contexto']]
    tool_calls = [log for log in resultado['logs'] if log['etapa'] == 'tool_call']

    print(f'## {nome}')
    print('Pergunta original:', pergunta)
    print('Query limpa:', resultado['query_limpa'])
    print('Contexto recuperado:', ids_contexto)
    print('Etapas executadas:', etapas)
    print('Tool calls:', json.dumps(tool_calls, ensure_ascii=False, indent=2))
    print('Resposta final:', resultado['resposta'])

    assert not any(char.isdigit() for char in resultado['query_limpa'])
    if esperado_produto:
        assert esperado_produto in ids_contexto
    if exige_tool:
        assert tool_calls, 'Esperava chamada da ferramenta validar_elegibilidade.'

    return resultado


## Caso 1: Capital de giro aprovado

Entrada esperada pelo loop: produto de capital de giro, empresa com 24 meses e faturamento de R$ 1.000.000.

In [3]:
caso_1 = resumir_resultado(
    nome='Capital de giro aprovado',
    pergunta='Quero crédito de giro, minha empresa tem 2 anos e fatura 1 milhão',
    esperado_produto='produto:capital_giro',
)

## Capital de giro aprovado
Pergunta original: Quero crédito de giro, minha empresa tem 2 anos e fatura 1 milhão
Query limpa: Quero crédito de giro, minha empresa tem anos e fatura milhão
Contexto recuperado: ['produto:capital_giro', 'contexto:capital_giro', 'produto:conta_garantida', 'contexto:liquidez']
Etapas executadas: ['query_original', 'query_limpa', 'retrieval', 'llm_primeira_resposta', 'tool_call', 'llm_resposta_final']
Tool calls: [
  {
    "etapa": "tool_call",
    "nome": "validar_elegibilidade",
    "argumentos": {
      "faturamento_cliente": 1000000,
      "idade_cliente": 24,
      "produto": "capital_giro"
    },
    "retorno": {
      "status": "aprovado"
    }
  }
]
Resposta final: Ótimo! Com base nos seus dados (faturamento de 1.000.000 e tempo de empresa de 2 anos ≈ 24 meses), o Capital de Giro Flex aderente foi validado.

Resumo rápido
- Produto: Capital de Giro Flex (chave capital_giro)
- Propósito: reforçar caixa, pagar fornecedores, cobrir despesas operacionais

## Caso 2: Capital de giro reprovado por faturamento

A empresa tem tempo suficiente, mas faturamento abaixo da regra deterministica do produto.

In [4]:
caso_2 = resumir_resultado(
    nome='Capital de giro reprovado por faturamento',
    pergunta='Quero crédito de giro, minha empresa tem 3 anos e fatura 100 mil',
    esperado_produto='produto:capital_giro',
)

## Capital de giro reprovado por faturamento
Pergunta original: Quero crédito de giro, minha empresa tem 3 anos e fatura 100 mil
Query limpa: Quero crédito de giro, minha empresa tem anos e fatura mil
Contexto recuperado: ['produto:capital_giro', 'contexto:capital_giro', 'contexto:liquidez', 'produto:conta_garantida']
Etapas executadas: ['query_original', 'query_limpa', 'retrieval', 'llm_primeira_resposta', 'tool_call', 'llm_resposta_final']
Tool calls: [
  {
    "etapa": "tool_call",
    "nome": "validar_elegibilidade",
    "argumentos": {
      "faturamento_cliente": 100000,
      "idade_cliente": 36,
      "produto": "capital_giro"
    },
    "retorno": {
      "status": "reprovado",
      "motivo": "Faturamento insuficiente"
    }
  }
]
Resposta final: Resumo da validação:
- Dados extraídos da pergunta: faturamento = 100.000 reais; tempo de empresa = 3 anos (36 meses); produto solicitado = Capital de Giro Flex (capital_giro).
- Resultado: não elegível para Capital de Giro Flex nest

## Caso 3: Dados insuficientes

A pergunta tem intencao de credito, mas nao informa faturamento nem tempo de empresa. O assistente deve pedir os dados faltantes em vez de inventar uma aprovacao.

In [5]:
caso_3 = resumir_resultado(
    nome='Dados insuficientes',
    pergunta='Quero uma linha para comprar máquinas novas para minha empresa',
    esperado_produto='produto:financiamento_maquinas',
    exige_tool=False,
)

## Dados insuficientes
Pergunta original: Quero uma linha para comprar máquinas novas para minha empresa
Query limpa: Quero uma linha para comprar máquinas novas para minha empresa
Contexto recuperado: ['produto:financiamento_maquinas', 'produto:antecipacao_recebiveis', 'contexto:capital_giro', 'produto:conta_garantida']
Etapas executadas: ['query_original', 'query_limpa', 'retrieval', 'llm_primeira_resposta']
Tool calls: []
Resposta final: Ótima opção. Para criar uma linha de financiamento de máquinas e equipamentos, o produto aderente no nosso contexto é o Financiamento de Maquinas e Equipamentos (produto chave: financiamento_maquinas).

Requisitos mínimos do produto:
- Faturamento mínimo: 1.200.000 (ano/periodo atual)
- Idade da empresa: 18 meses (ou mais)

Para eu checar sua elegibilidade, preciso que você me informe:
- Faturamento mensal atual da empresa (em reais)
- Idade da empresa em meses

Se preferir, posso também apresentar alternativas caso não alcance esses requisitos:
- A